In [1]:
!pip install pvlib

In [2]:
width_m, depth_m, height_m = 25, 25, 81
window_wall_ratio = 0.90
T_indoor_C = 23
T_outdoor_C = 35
U_wall = 0.25
U_roof = 0.20
U_glass = 1.4
ACH = 0.8
air_density = 1.2
air_cp = 1005
n_people = 300
person_heat_W = 200
light_total = 7
n_lights = 6500
electricity_price_eur_kwh = 0.20
cooling_cop = 2.5
heating_cop = 3.5

In [3]:
footprint_m2 = width_m * depth_m
dT = T_outdoor_C - T_indoor_C
facade_area_by_orientation = {
    "North": width_m * height_m, "South": width_m * height_m,
    "East": depth_m * height_m, "West": depth_m * height_m,
}
glazing_area_by_orientation = {k: v * window_wall_ratio for k, v in facade_area_by_orientation.items()}
glazing_area = sum(glazing_area_by_orientation.values())
wall_area = 2 * (width_m + depth_m) * height_m * (1 - window_wall_ratio)
roof_area = footprint_m2
volume_m3 = footprint_m2 * height_m
Q_people = n_people * person_heat_W
Q_light = light_total * n_lights
Q_wall = U_wall * wall_area * dT
Q_roof = U_roof * roof_area * dT
Q_glass_cond = U_glass * glazing_area * dT
Q_vent = ACH * volume_m3 * air_density * air_cp * dT / 3600
Q_constant = Q_people + Q_light + Q_wall + Q_roof + Q_glass_cond + Q_vent
print(f"Glazing area: {glazing_area:.0f} m2, Q_constant: {Q_constant:.0f} W")

Glazing area: 7290 m2, Q_constant: 394712 W


In [4]:
import numpy as np

tau_e_0 = {0: 0.450, 25: 0.416, 50: 0.364, 100: 0.213}
p = 2

def karlsson_roos_tau(theta_deg, tau0, q, p=2):
    if theta_deg >= 90:
        return 0.0
    z = theta_deg / 90
    a = 8
    b = 0.25 / q
    c = 1 - a - b
    alpha = 5.2 + 0.7*q
    beta = 2
    gamma = (5.26 + 0.06*p) + (0.73 + 0.14*q)
    factor = 1 - a*(z**alpha) - b*(z**beta) - c*(z**gamma)
    return tau0 * factor

q_guess = {0: 2.19, 25: 2.15, 50: 2.24, 100: 3.92}

In [5]:
from scipy.integrate import quad

def tau_e_diffuse(state):
    tau0 = tau_e_0[state]
    q = q_guess[state]
    def integrand(phi_rad):
        phi_deg = np.degrees(phi_rad)
        return karlsson_roos_tau(phi_deg, tau0, q, p) * np.cos(phi_rad) * np.sin(phi_rad)
    integral, _ = quad(integrand, 0, np.pi/2)
    return 2 * integral

tau_e_dif = {state: tau_e_diffuse(state) for state in [0, 25, 50, 100]}

In [6]:
# Piccolo, Marino, Nucara, Pietrafesa (2018), Energy & Buildings 165, 390-398, Tabela 2
rho_e_light = 0.099
rho_e_dark = 0.083
rho_e = {state: rho_e_light + (state/100)*(rho_e_dark - rho_e_light) for state in [0, 25, 50, 100]}

In [7]:
Fi = 0.258

def alpha_e_direct(theta_deg, state):
    tau = karlsson_roos_tau(theta_deg, tau_e_0[state], q_guess[state], p=p)
    return 1 - tau - rho_e[state]

def qi_direct(theta_deg, state):
    return Fi * alpha_e_direct(theta_deg, state)

def g_direct(theta_deg, state):
    tau = karlsson_roos_tau(theta_deg, tau_e_0[state], q_guess[state], p=p)
    return tau + qi_direct(theta_deg, state)

In [8]:
def alpha_e_diffuse(state):
    return 1 - tau_e_dif[state] - rho_e[state]

def qi_diffuse(state):
    return Fi * alpha_e_diffuse(state)

def g_diffuse(state):
    return tau_e_dif[state] + qi_diffuse(state)

g_dif = {state: g_diffuse(state) for state in [0, 25, 50, 100]}

In [9]:
import pvlib

FILEPATH_EPW = "SVN_LJ_Ljubljana-Bezigrad.140150_TMYx.2007-2021.epw"
epw_data, epw_meta = pvlib.iotools.read_epw(FILEPATH_EPW)
LAT = epw_meta['latitude']
LON = epw_meta['longitude']
solar_position = pvlib.solarposition.get_solarposition(epw_data.index, LAT, LON)
print(f"Lokacija: {epw_meta['city']}, {len(epw_data)} ur")

Lokacija: Ljubljana-Bezigrad, 8760 ur


In [10]:
facade_azimuths = {"North": 0, "East": 90, "South": 180, "West": 270}
dni_extra = pvlib.irradiance.get_extra_radiation(epw_data.index)

facade_irradiance = {}
for facade, az in facade_azimuths.items():
    total_irrad = pvlib.irradiance.get_total_irradiance(
        surface_tilt=90, surface_azimuth=az,
        solar_zenith=solar_position['apparent_zenith'],
        solar_azimuth=solar_position['azimuth'],
        dni=epw_data['dni'], ghi=epw_data['ghi'], dhi=epw_data['dhi'],
        dni_extra=dni_extra, model='perez'
    )
    aoi = pvlib.irradiance.aoi(
        surface_tilt=90, surface_azimuth=az,
        solar_zenith=solar_position['apparent_zenith'],
        solar_azimuth=solar_position['azimuth']
    )
    facade_irradiance[facade] = {
        'aoi': aoi,
        'poa_direct': total_irrad['poa_direct'].clip(lower=0),
        'poa_diffuse_total': (total_irrad['poa_diffuse'] + total_irrad.get('poa_ground_diffuse', 0)).clip(lower=0),
    }

In [11]:
q_sol_results = {}
for state in [0, 25, 50, 100]:
    q_sol_by_facade = {}
    for facade in facade_azimuths:
        aoi = facade_irradiance[facade]['aoi']
        poa_direct = facade_irradiance[facade]['poa_direct']
        poa_diffuse = facade_irradiance[facade]['poa_diffuse_total']
        g_direct_hourly = aoi.apply(lambda theta: g_direct(theta, state) if theta < 90 else 0)
        q_sol_by_facade[facade] = g_direct_hourly * poa_direct + g_dif[state] * poa_diffuse
    q_sol_results[state] = q_sol_by_facade

In [12]:
power_density_by_state = {0: 1.00, 25: 0.75, 50: 0.50, 100: 0.00}

In [13]:
percent_time_at_0 = 25    # <-- rocno spreminjaj
percent_time_at_25 = 25   # <-- rocno spreminjaj
percent_time_at_50 = 25   # <-- rocno spreminjaj
percent_time_at_100 = 25  # <-- rocno spreminjaj

assert percent_time_at_0+percent_time_at_25+percent_time_at_50+percent_time_at_100 == 100

activation_mix = {
    0: percent_time_at_0/100, 25: percent_time_at_25/100,
    50: percent_time_at_50/100, 100: percent_time_at_100/100,
}
print("Aktivacijska mesanica:", activation_mix)

Aktivacijska mesanica: {0: 0.25, 25: 0.25, 50: 0.25, 100: 0.25}


In [14]:
baseline_solar_hourly = sum(q_sol_results[0][facade] * glazing_area_by_orientation[facade] for facade in facade_azimuths)

weighted_solar_hourly = sum(
    sum(q_sol_results[state][facade] * glazing_area_by_orientation[facade] for facade in facade_azimuths) * activation_mix[state]
    for state in [0, 25, 50, 100]
)

occupied_hours_mask = (epw_data.index.hour >= 7) & (epw_data.index.hour < 22)
Q_internal_hourly = Q_constant * (occupied_hours_mask.astype(float)*1.0 + (~occupied_hours_mask).astype(float)*0.35)

In [15]:
def annual_energy_breakdown(solar_hourly):
    total_load_hourly = Q_internal_hourly + solar_hourly

    cooling_wh_hourly = total_load_hourly.clip(lower=0)
    cooling_kwh_annual = cooling_wh_hourly.sum() / 1000
    cooling_electricity_kwh = cooling_kwh_annual / cooling_cop

    heating_wh_hourly = (-total_load_hourly).clip(lower=0)
    heating_kwh_annual = heating_wh_hourly.sum() / 1000
    heating_electricity_kwh = heating_kwh_annual / heating_cop

    return {
        'cooling_electricity_kwh': cooling_electricity_kwh,
        'cooling_cost_eur': cooling_electricity_kwh * electricity_price_eur_kwh,
        'heating_electricity_kwh': heating_electricity_kwh,
        'heating_cost_eur': heating_electricity_kwh * electricity_price_eur_kwh,
    }

baseline_breakdown = annual_energy_breakdown(baseline_solar_hourly)
weighted_breakdown = annual_energy_breakdown(weighted_solar_hourly)

cooling_savings_eur = baseline_breakdown['cooling_cost_eur'] - weighted_breakdown['cooling_cost_eur']
heating_extra_cost_eur = weighted_breakdown['heating_cost_eur'] - baseline_breakdown['heating_cost_eur']

print(f"Prihranek pri hlajenju: {cooling_savings_eur:,.0f} EUR/leto")
print(f"Dodaten strosek ogrevanja: {heating_extra_cost_eur:,.0f} EUR/leto")
print(f"NETO prihranek (energija): {cooling_savings_eur - heating_extra_cost_eur:,.0f} EUR/leto")

Prihranek pri hlajenju: 30,683 EUR/leto
Dodaten strosek ogrevanja: 0 EUR/leto
NETO prihranek (energija): 30,683 EUR/leto


In [16]:
weighted_power_density = sum(power_density_by_state[state]*activation_mix[state] for state in [0,25,50,100])
window_operating_kwh = weighted_power_density * glazing_area * len(epw_data) / 1000
window_operating_cost_eur = window_operating_kwh * electricity_price_eur_kwh
print(f"Strosek delovanja stekla: {window_operating_cost_eur:,.0f} EUR/leto")

Strosek delovanja stekla: 7,184 EUR/leto


In [17]:
net_annual_savings_eur = (cooling_savings_eur - heating_extra_cost_eur) - window_operating_cost_eur

cost_standard_glass_eur_m2 = 300
cost_spd_glass_eur_m2 = 480
fixed_installation_cost_eur = 10000
avoided_blinds_cost_eur_m2 = 150
avoided_blinds_total_eur = avoided_blinds_cost_eur_m2 * glazing_area
extra_cost_per_m2 = cost_spd_glass_eur_m2 - cost_standard_glass_eur_m2
gross_extra_investment_eur = extra_cost_per_m2 * glazing_area + fixed_installation_cost_eur
total_extra_investment_eur = gross_extra_investment_eur - avoided_blinds_total_eur

print(f"NETO letni prihranek: {net_annual_savings_eur:,.0f} EUR")
print(f"Neto investicija: {total_extra_investment_eur:,.0f} EUR")

if total_extra_investment_eur > 0 and net_annual_savings_eur > 0:
    print(f"Doba amortizacije: {total_extra_investment_eur/net_annual_savings_eur:.1f} let")
elif total_extra_investment_eur <= 0:
    print("SPD je net cenejsi -- takojsen prihranek.")
else:
    print("Investicija se ne povrne.")

NETO letni prihranek: 23,498 EUR
Neto investicija: 228,700 EUR
Doba amortizacije: 9.7 let
